# Milestone 3 tour: experience that remains causal

Milestone 3 makes earlier work useful to later proposers without turning memory into ambient authority. This tour follows the four slices in order: **M3.1 temporal graph**, **M3.2 pushed context**, **M3.3 audited pull access**, and **M3.4 declared derivation and comparison**.

The deterministic experiment deliberately produces two failed evaluations. Memory-aware variants can then reuse those failures to reach `10`; the no-memory variant cannot.

## 1. Run the same experiment four ways

Task, seed, proposer, evaluator, chain search, random seed, and task/run budgets stay fixed. Only the memory declaration changes.

In [1]:
from pathlib import Path
import runpy

from meta_evolve.application import ExperienceQuery, replay_experience_graph

path = Path("main.py")
if not path.exists():
    path = Path("examples/03_experience_reuse/main.py")
demo = runpy.run_path(path)
results = demo["run_experiment"]()
print("variants:", tuple(results))

variants: ('no-memory', 'push-only', 'pull-only', 'combined')


In [2]:
for name, (result, _) in results.items():
    summary = result.summary()
    print(
        name,
        f"score={summary.primary_score}",
        f"task_usage={summary.usage}",
        f"memory_usage={summary.experience_usage}",
    )

no-memory score=0 task_usage=Usage(evaluations=4, trials=3, tokens=0, wall_seconds=0.0) memory_usage=RunExperienceUsage(context=ContextUsage(records=0, chars=0), pull=ExperienceUsage(operations=0, results=0, records=0, chars=0))
push-only score=10 task_usage=Usage(evaluations=4, trials=3, tokens=0, wall_seconds=0.0) memory_usage=RunExperienceUsage(context=ContextUsage(records=3, chars=3295), pull=ExperienceUsage(operations=0, results=0, records=0, chars=0))
pull-only score=10 task_usage=Usage(evaluations=4, trials=3, tokens=0, wall_seconds=0.0) memory_usage=RunExperienceUsage(context=ContextUsage(records=0, chars=0), pull=ExperienceUsage(operations=7, results=8, records=20, chars=20559))
combined score=10 task_usage=Usage(evaluations=4, trials=3, tokens=0, wall_seconds=0.0) memory_usage=RunExperienceUsage(context=ContextUsage(records=3, chars=3295), pull=ExperienceUsage(operations=7, results=8, records=20, chars=20855))


## 2. M3.1 — the temporal experience graph

The graph is a read model over committed trials and records. Nodes are causal occurrences, not deduplicated payloads. An `as_of` boundary can reveal the past without leaking future trials.

In [3]:
combined, storage = results["combined"]
events = storage.events.read(combined.id)
graph = replay_experience_graph(combined.id, events)
query = ExperienceQuery(task_id=graph.task_id, run_id=combined.id)
state = graph.state_at(query)

for node in state.nodes:
    kinds = tuple(record.kind for record in node.records)
    print(f"step={node.logical_step} records={kinds}")
print("M3.1 frontier:", [node.logical_step for node in graph.frontier(query)])

step=0 records=('trial', 'proposal', 'artifact', 'outcome', 'evaluation')
step=1 records=('decision', 'decision_explanation', 'context_selection', 'experience_grant', 'experience_attempt', 'experience_success', 'proposal', 'artifact', 'trial', 'outcome', 'evaluation')
step=2 records=('decision', 'decision_explanation', 'context_selection', 'experience_grant', 'experience_attempt', 'experience_success', 'experience_attempt', 'experience_success', 'proposal', 'artifact', 'trial', 'outcome', 'evaluation')
step=3 records=('decision', 'decision_explanation', 'context_selection', 'experience_grant', 'experience_attempt', 'experience_success', 'experience_attempt', 'experience_success', 'experience_attempt', 'experience_success', 'experience_attempt', 'experience_success', 'proposal', 'artifact', 'trial', 'outcome', 'evaluation')
M3.1 frontier: [3]


In [4]:
historical = graph.state_at(ExperienceQuery(
    task_id=graph.task_id,
    run_id=combined.id,
    as_of=2,
))
print("visible steps as_of=2:", [node.logical_step for node in historical.nodes])
print("frontier steps:", [node.logical_step for node in state.nodes])

visible steps as_of=2: [0, 1, 2]
frontier steps: [0, 1, 2, 3]


## 3. M3.2 — policy-pushed context

A context policy chooses a bounded, exact bundle before the proposer runs. The bundle stores structured records, a canonical rendering, a reason, truncation, and usage. It is replayed rather than selected again.

In [5]:
projection = combined._projection()
for selected in projection.context_selections:
    refs = tuple(record.ref.kind for record in selected.bundle.records)
    print(
        f"step={selected.bundle.records[-1].logical_step if refs else '-'}",
        f"reason={selected.bundle.reason!r}",
        f"refs={refs}",
        f"usage={selected.bundle.usage}",
    )
print("M3.2 declaration:", projection.context_declaration.spec)

step=- reason='failed ancestor evaluations' refs=() usage=ContextUsage(records=0, chars=0)
step=1 reason='failed ancestor evaluations' refs=('evaluation',) usage=ContextUsage(records=1, chars=1098)
step=2 reason='failed ancestor evaluations' refs=('evaluation', 'evaluation') usage=ContextUsage(records=2, chars=2197)
M3.2 declaration: ContextSpec(policy=PolicyRef(name='failed-ancestors', version='1', origin='local'), max_records=20, max_chars=8000)


## 4. M3.3 — agent-directed pull access

The reader lets a proposer search, open, traverse, and compare within a strictly-prior grant. Every attempt is charged and resolves to success, opaque denial, or content-free exhaustion. Usage is cumulative inside each grant.

In [6]:
for issued in projection.experience_grants:
    operations = projection.operations_for(issued.grant.trial_id)
    print(f"grant as_of={issued.grant.as_of} operations={len(operations)}")
    for attempt, resolution in operations:
        print(
            " ",
            attempt.request.operation,
            type(resolution).__name__,
            projection.experience_usage_for(attempt.trial_id),
        )
print("M3.3 declaration:", projection.experience_declaration.spec)

grant as_of=0 operations=1
  search ExperienceOperationSucceeded ExperienceUsage(operations=1, results=0, records=0, chars=0)
grant as_of=1 operations=2
  search ExperienceOperationSucceeded ExperienceUsage(operations=2, results=2, records=2, chars=2196)
  open ExperienceOperationSucceeded ExperienceUsage(operations=2, results=2, records=2, chars=2196)
grant as_of=2 operations=4
  search ExperienceOperationSucceeded ExperienceUsage(operations=4, results=6, records=18, chars=18659)
  open ExperienceOperationSucceeded ExperienceUsage(operations=4, results=6, records=18, chars=18659)
  open ExperienceOperationSucceeded ExperienceUsage(operations=4, results=6, records=18, chars=18659)
  compare ExperienceOperationSucceeded ExperienceUsage(operations=4, results=6, records=18, chars=18659)
M3.3 declaration: ExperienceSpec(max_operations=20, max_results=20, max_records=100, max_chars=32000)


## 5. M3.4 — declare what actually caused the proposal

Observation and use are distinct. `derived_from` must be a unique ordered subsequence of the committed push-then-pull ledger. Valid edges survive failed evaluations and typed proposer failures; unauthorized declarations become policy failures.

In [7]:
for node in state.nodes[1:]:
    print(
        f"step={node.logical_step}",
        "derived_from=",
        tuple(reference.kind for reference in node.derived_from),
    )

baseline = results["no-memory"][0]
comparison = baseline.compare(combined)
print("M3.4 winner score:", comparison.winner.primary_score)
print("same context:", comparison.same_context_declaration)
print("same pull:", comparison.same_experience_declaration)

step=1 derived_from= ()
step=2 derived_from= ('evaluation',)
step=3 derived_from= ('evaluation', 'evaluation', 'trial', 'trial')
M3.4 winner score: 10
same context: False
same pull: False


## What to keep in your head

- **Graph:** what committed experience existed at a logical time.
- **Push:** what a policy placed in front of the proposer.
- **Pull:** what the proposer chose to inspect through a bounded capability.
- **Derivation:** which observed references the proposer says it actually used.
- **Evaluation:** still independent of proposer authority.

The example is a small deterministic illustration, not broad scientific evidence. Its purpose is to make the authority and replay contracts tangible.